<a href="https://colab.research.google.com/github/Magistrate-dot/ML-AI-project-_Minecraft_Object_Detection/blob/main/Minecraft_Mob_Detection_model_Comparison_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prep

In [ ]:
!pip install -q ultralytics huggingface_hub scikit-learn
import os
import json
import shutil
from collections import defaultdict
from sklearn.model_selection import train_test_split

from huggingface_hub import snapshot_download
DATASET_PATH = "/content/minecraft_dataset"

repo_dir = snapshot_download(
    repo_id="twoturtles/minecraft-mobs",
    repo_type="dataset"
)

print(repo_dir)

!wget -O test_image.jpg "https://raw.githubusercontent.com/Magistrate-dot/ML-AI-project-_Minecraft_Object_Detection/main/Test_mob.jpg"


for split in ["train", "valid"]:
    os.makedirs(f"{DATASET_PATH}/{split}/images", exist_ok=True)
    os.makedirs(f"{DATASET_PATH}/{split}/labels", exist_ok=True)

with open(os.path.join(repo_dir, "annotations.json"), "r") as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]

annotations_by_image = defaultdict(list)

for ann in annotations:
    annotations_by_image[ann["image_id"]].append(ann)

train_images, valid_images = train_test_split(
    images,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Training images:", len(train_images))
print("Validation images:", len(valid_images))

def process_split(image_list, split_name):

    for img in image_list:

        image_id = img["id"]
        filename = img["file_name"]

        src = os.path.join(repo_dir, "images", filename)
        dst = os.path.join(DATASET_PATH, split_name, "images", filename)

        shutil.copy(src, dst)

        label_file = os.path.join(
            DATASET_PATH,
            split_name,
            "labels",
            filename.replace(".png", ".txt")
        )

        with open(label_file, "w") as f:

            for ann in annotations_by_image.get(image_id, []):

                x, y, w, h = ann["bbox"]

                xc = (x + w / 2) / img["width"]
                yc = (y + h / 2) / img["height"]
                wn = w / img["width"]
                hn = h / img["height"]

                cls = ann["category_id"]

                f.write(
                    f"{cls} {xc} {yc} {wn} {hn}\n"
                )
process_split(train_images, "train")
process_split(valid_images, "valid")

yaml_text = f"""
path: {DATASET_PATH}

train: train/images
val: valid/images

names:
  0: chicken
  1: cow
  2: creeper
  3: enderman
  4: pig
  5: sheep
  6: skeleton
  7: spider
  8: zombie
"""

with open("data.yaml", "w") as f:
    f.write(yaml_text)

print("Train images:", len(os.listdir(f"{DATASET_PATH}/train/images")))
print("Train labels:", len(os.listdir(f"{DATASET_PATH}/train/labels")))

print("Validation images:", len(os.listdir(f"{DATASET_PATH}/valid/images")))
print("Validation labels:", len(os.listdir(f"{DATASET_PATH}/valid/labels")))



In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

# Count annotations per class
class_counts = Counter()

for ann in annotations:
    class_counts[ann["category_id"]] += 1

# Class neames
class_names = {
    0: "chicken",
    1: "cow",
    2: "creeper",
    3: "enderman",
    4: "pig",
    5: "sheep",
    6: "skeleton",
    7: "spider",
    8: "zombie"
}

# Create DataFrame
df_classes = pd.DataFrame({
    "Class": [class_names[i] for i in sorted(class_counts.keys())],
    "Annotations": [class_counts[i] for i in sorted(class_counts.keys())]
})

print(df_classes)

# Model

In [ ]:
from ultralytics import YOLO

nano = YOLO("yolo11n.pt")
nano.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
)

small = YOLO("yolo11s.pt")
small.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
)

# Analytics

In [ ]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import os

print("Extracting final metrics and training history...")

# 1. Grab final validation metrics (keeps your terminal printout working!)
nano_metrics = YOLO("/content/runs/detect/train/weights/best.pt").val(split="val")
small_metrics = YOLO("/content/runs/detect/train-2/weights/best.pt").val(split="val")

# Generate and print the data frame summary
comparison = pd.DataFrame({
    "Model": ["YOLO11 Nano", "YOLO11 Small"],
    "Precision": [nano_metrics.box.mp, small_metrics.box.mp],
    "Recall": [nano_metrics.box.mr, small_metrics.box.mr],
    "mAP50": [nano_metrics.box.map50, small_metrics.box.map50],
    "mAP50-95": [nano_metrics.box.map, small_metrics.box.map]
})

print("\n=== FINAL MODEL PERFORMANCE COMPARISON ===")
print(comparison)
comparison.to_csv("final_50_epoch_comparison.csv", index=False)


# create line graph
nano_csv_path = "/content/runs/detect/train/results.csv"
small_csv_path = "/content/runs/detect/train-2/results.csv"

if os.path.exists(nano_csv_path) and os.path.exists(small_csv_path):
    # Load history files
    df_nano = pd.read_csv(nano_csv_path)
    df_small = pd.read_csv(small_csv_path)

    # Clean up column names
    df_nano.columns = df_nano.columns.str.strip()
    df_small.columns = df_small.columns.str.strip()

    # Set up the line graph layout
    plt.figure(figsize=(10, 6))

    # Plot Nano Curve (Solid Green Line)
    plt.plot(df_nano['epoch'], df_nano['metrics/mAP50(B)'],
             label='YOLO11 Nano', color='#4CAF50', linewidth=2.5, linestyle='-')

    # Plot Small Curve (Dashed Blue Line)
    plt.plot(df_small['epoch'], df_small['metrics/mAP50(B)'],
             label='YOLO11 Small', color='#2196F3', linewidth=2.5, linestyle='--')

    # Style the graph
    plt.title("Model Convergence: Validation Accuracy (mAP50) over 50 Epochs", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Epochs", fontsize=12)
    plt.ylabel("Validation Accuracy (mAP50 Score)", fontsize=12)

    plt.xlim(1, 50)
    plt.ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(fontsize=12, loc='lower right')
    plt.show() # Display the line graph

    # Save a file for downloading
    plt.savefig("model_accuracy_line_graph.png", dpi=300, bbox_inches='tight')
    print("\n Line graph generated successfully as 'model_accuracy_line_graph.png'!")
else:
    print("\n Error: Could not find 'results.csv' in /train/ or /train-2/ folders.")

# Plot the mAP50 performance side-by-side
plt.figure(figsize=(6, 4))
plt.bar(comparison["Model"], comparison["mAP50"], color=['#4CAF50', '#2196F3'])
plt.title("Minecraft Mob Detection: 50-Epoch Model Comparison (mAP50)")
plt.ylabel("mAP50 Score")
plt.ylim(0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# Test

In [ ]:
import os
from ultralytics import YOLO
from IPython.display import Image, display

# 1. Path to your uploaded JPG image inside the content folder
image_path = "/content/test_image.jpg"

# Load both models with clear names using their correct save paths
small_model = YOLO("/content/runs/detect/train-2/weights/best.pt")
nano_model  = YOLO("/content/runs/detect/train/weights/best.pt")

print(f"Analyzing '{os.path.basename(image_path)}' for Minecraft mobs with both models...")

# Run prediction with the small_model and save visually
results_small = small_model.predict(source=image_path, conf=0.25, save=True)

# Grab the output paths dynamically for the small model
saved_dir_small = results_small[0].save_dir
base_name_small = os.path.basename(results_small[0].path)
output_visual_path_small = os.path.join(saved_dir_small, base_name_small)

print("\n DETECTION COMPLETE (YOLO11 Small)! Here is what the Small AI found:")
display(Image(filename=output_visual_path_small))

# Run prediction with the nano_model and save visually
results_nano = nano_model.predict(source=image_path, conf=0.25, save=True)

# Grab the output paths dynamically for the nano model
saved_dir_nano = results_nano[0].save_dir
base_name_nano = os.path.basename(results_nano[0].path)
output_visual_path_nano = os.path.join(saved_dir_nano, base_name_nano)

print("\n DETECTION COMPLETE (YOLO11 Nano)! Here is what the Nano AI found:")
display(Image(filename=output_visual_path_nano))